v61  
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
1.  **Strict Date Logic:** No more "Time Travel" bugs.
2.  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
3.  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
4.  **Performance Workers:** Math is separated from orchestration.


v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


v57, v58  
added marco subplotsThe macro regime framework is now fully documented with:
- Trend (SMA200 deviation) → Where we are in the cycle  
- Trend Velocity (Z) → How fast we're moving relative to normal
- VIX-Z → Market fear/complacency levels  

v56  

- De-coupled features_df and macro_df
- generate_features and audit_feature_engineering_integrity use GLOBAL_SETTINGS

v55  
Added
- audit_feature_engineering_integrity (check calculation in features_df)  

These are the metrics in plot  
- --- 1. LEGACY / SANITY CHECKS ---
- "Price Gain": lambda obs: QuantUtils.calculate_gain(obs["lookback_close"]),
- "Sharpe": lambda obs: QuantUtils.calculate_sharpe(obs["lookback_returns"]),
- "Sharpe (ATRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["atrp"]
    ),
- "Sharpe (TRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["trp"]
    ),
- --- 2. NEW QUANT METRICS ---
- "Momentum (21d)": lambda obs: obs["mom_21"],
- "Information Ratio (IR)": lambda obs: obs["ir_63"],  # Kept this one
- "Consistency (WinRate)": lambda obs: obs["consistency"],
- "Oversold (RSI)": lambda obs: -obs["rsi"],
- "Dip Buyer (Drawdown)": lambda obs: -obs["dd_21"],
- "Low Volatility": lambda obs: -obs["atrp"],







v54
-  **Replaced plot_walk_forward_analyzer with create_walk_forward_analyzer**


v53  
Looking at this registry with a quant lens, the list is **comprehensive but bloated**—we have **momentum measured five times under different names** (roc₁, roc₃, roc₅, roc₁₀, roc₂₁ and their negative twins “Pullback”).  
That’s **10 slots** telling us almost the same story at slightly different lags; in a rank-based engine they will **crowd the signal space** and inflate turnover without adding IC.

Duplicate / redundant cluster  
- Momentum 1 D ↔ Pullback 1 D (perfect mirror)  
- Same for 3 D, 5 D, 10 D, 21 D.  
**Keep one side only**—momentum is enough; the portfolio constructor can always **reverse the rank** if it wants “oversold”.

Close cousins that can be merged  
- “Sharpe” vs “Sharpe (ATRP)” – both are return / vol; keep **ATRP version** because it is regime-aware and smoother.  
- “RVol” vs “Vol_Regime” – both capture vol expansion; keep the **longer-memory one** (Vol_Regime) and drop the intraday snapshot.

Gaps that matter to a quant  
1. **Consistency sensor**: nowhere do we ask “how often did the ticker close higher than it opened?” – add **5-day win-rate** or **up-day hit-ratio**.  
2. **Risk-adjusted intraday strength**: no **Sharpe(on-balance volume)** or **volume-momentum efficiency**; OBV_Score is raw, not risk-scaled.  
3. **Benchmark-relative consistency**: “Alpha (RelStrength)” is cumulative; add **rolling information ratio vs SPY** to catch *sustained* alpha, not one gap.  
4. **Tail flag**: no **skew** or **max-drawdown** metric; a single 20 % gap stock can poison the book.  
5. **Macro regime overlay**: no **beta-to-SPY** or **correlation-break** sensor; mid-2022 macro swings showed that low-beta names behaved like a different asset class.

Recommended minimal clean set (≤ 12 metrics)

1. Sharpe(ATRP) – strategic anchor  
2. Momentum 21 D – slow trend  
3. Momentum 5 D – fast trend  
4. 5-day win-rate – consistency  
5. RSI(Trend) – strength confirmation  
6. OBV_Score – volume conviction  
7. Vol_Regime – vol expansion filter  
8. Alpha(RelStrength) 63-day IR – benchmark consistency  
9. Max 21-day drawdown – tail guard  
10. Beta-to-SPY – macro regime tag  

Drop everything else; the freed-up slots reduce collinearity, cut turnover, and leave head-room for **interaction terms** (e.g. momentum × consistency) that actually add orthogonal signal.



Below is a single, fully-vectorised block that adds the **five gap metrics** to your existing MultiIndex OHLCV frame.  
It never loops over tickers; everything is done with `groupby(level=0).rolling(...)` so it runs in C-speed and keeps the same index.

```python
import pandas as pd
import numpy as np

# ----------  CONFIG  -------------------------------------------------
LKB_RET   = 21          # look-back for return-based metrics
LKB_CONS  = 5           # consistency window (days)
LKB_IR    = 63          # IR window
LKB_BETA  = 63          # beta window
LKB_TAIL  = 21          # max-drawdown window
BENCH     = 'SPY'       # ticker that exists in your universe
# ---------------------------------------------------------------------

# 1.  DAILY RETURNS ----------------------------------------------------
df['ret'] = df.groupby(level=0)['Adj Close'].pct_change()

# 2.  CONSISTENCY SENSOR  (5-day win-rate) -----------------------------
df['up']  = df['ret'].gt(0).astype(int)
df['consistency_5d'] = (df.groupby(level=0)['up']
                          .rolling(LKB_CONS).mean()
                          .reset_index(level=0, drop=True))

# 3.  BENCHMARK-RELATIVE CONSISTENCY  (63-day IR vs SPY) ---------------
# need benchmark return
bench_ret = df.xs(BENCH, level=0)['ret'].rename('bench_ret')
df = df.join(bench_ret, how='left')          # broadcast to all tickers

df['active'] = df['ret'] - df['bench_ret']
g = df.groupby(level=0)
active_mean  = g['active'].rolling(LKB_IR).mean()
active_std   = g['active'].rolling(LKB_IR).std()
df['IR_63d'] = active_mean / active_std      # Information Ratio

# 4.  TAIL FLAG  (21-day max drawdown) ---------------------------------
roll_max = g['Adj Close'].rolling(LKB_TAIL).max()
dd = (df['Adj Close'] - roll_max) / roll_max
df['max_dd_21d'] = dd.groupby(level=0).rolling(LKB_TAIL).min()

# 5.  MACRO REGIME OVERLAY  (beta to SPY) ------------------------------
cov  = g['ret'].rolling(LKB_BETA).cov(df['bench_ret'])
var  = df['bench_ret'].groupby(level=0).rolling(LKB_BETA).var()
df['beta_SPY'] = cov / var

# 6.  RISK-ADJUSTED INTRADAY STRENGTH  (OBV Sharpe) --------------------
# OBV
df['close_chg'] = df.groupby(level=0)['Adj Close'].diff()
df['vol_dir']   = np.where(df['close_chg'] > 0,  df['Volume'],
                   np.where(df['close_chg'] < 0, -df['Volume'], 0))
df['obv'] = df.groupby(level=0)['vol_dir'].cumsum()

# OBV return & vol
df['obv_ret'] = df.groupby(level=0)['obv'].pct_change()
obv_mean = g['obv_ret'].rolling(LKB_RET).mean()
obv_std  = g['obv_ret'].rolling(LKB_RET).std()
df['OBV_Sharpe_21d'] = obv_mean / obv_std

# drop helper columns --------------------------------------------------
df.drop(columns=['up','bench_ret','active','close_chg','vol_dir'], inplace=True)
```

After the block you have five new columns:

- `consistency_5d`      – 5-day win-rate (0-1)  
- `IR_63d`              – 63-day Information Ratio vs SPY  
- `max_dd_21d`          – 21-day maximum drawdown (≤ 0)  
- `beta_SPY`            – rolling beta to SPY  
- `OBV_Sharpe_21d`      – OBV risk-adjusted momentum  

All are aligned to the original MultiIndex and ready to be ranked or z-scored inside your Alpha Engine.

v52  
- **Cascase Filter results `AGREED` with bot_v54i.ipynb**
- **Cascade Filter works with df_ohlcv_subset**
- **verify_engine_results_short_form**
- **verify_engine_results_long_form**
-  **The Temporal Alignment Fix:** We synchronized the "Reward" (Returns) and "Risk" (Volatility) by implementing the $N-1$ denominator logic. This ensures that Day 1's volatility no longer dilutes your Sharpe scores.
-  **The Event-Driven Re-normalization:** We verified that the Engine correctly resets capital and weights at the start of the Holding period, giving you an accurate "Fresh Start" performance metric.
-  **The Double-Blind Verification:** We proved that the Engine's True Range (TRP) math is flawless by recreating it from raw High/Low/Close data and achieving an 8-decimal match.
-  **Mathematical Fortification:** We centralized all logic into a polymorphic `QuantUtils` kernel that handles both single-portfolio reports and whole-universe rankings with built-in numerical safety.
-  **Volatility Evolution:** We successfully added `TRP` (True Range Percent) and the `Sharpe (TRP)` metric, giving you a raw, high-frequency alternative to the smoothed ATR.
-  **Data Integrity:** We implemented the "Momentum Collapse" tripwire (`verify_ranking_integrity`) to ensure that your risk-adjusted rankings never accidentally devolve into simple price momentum.
-  **The "Audit Pack" Architecture:** We collapsed fragmented results into a single, atomic container, ensuring that your inputs, results, and debug data are always perfectly synchronized.
-  **Total Transparency:** We replaced scattered CSV files with a unified **Excel Audit Report**, allowing for 1-to-1 manual verification of every calculation in the system.



v51

UNDO v50, Calculate Sharpe(ATR) using mean over lookback period.  

Comment out ``# --- PINPOINT START: ATRP SWITCH ---`` in function ``_select_tickers`` can switch between ``Averaged ATRP over lookback period`` and ``Current ATRP``  
    # --- PINPOINT START: ATRP SWITCH ---  
    # To switch between Old (Averaged ATRP) and New (Current ATRP):  
    # 1. Comment out the logic you DON'T want.  
    # 2. Uncomment the logic you DO want.  


v50

Ticker selection based on atrp_value_for_obs based on decision day, was based on average over lookback period. 

v48  
### Summary of what you just accomplished:
1.  **Strict Math:** `QuantUtils` now contains an `assert` that prevents any dev (or AI) from filling the first day with 0.0.
2.  **Semantic Protection:** Variables are now named `returns_WITH_BOUNDARY_NAN`, signaling to the AI that the Null value is part of its identity.
3.  **Complete SOLID Separation:** The Engine CONDUCTS the simulation, while `QuantUtils` CALCULATES the results. They no longer share logic.

**1. Data Flow of `plot_walk_forward_analyzer`**
The function acts as a **UI wrapper** around the `AlphaEngine` class. The flow is:
1.  **Input:** User selects parameters (Dates, Lookback, Strategy).
2.  **State Construction:** `AlphaEngine` slices the historical data (`df_ohlcv`, `df_atrp`) up to the `decision_date`.
3.  **Policy Execution (Hardcoded):** The engine applies the logic (e.g., `METRIC_REGISTRY['Sharpe']`) to rank stocks based *only* on the Lookback window.
4.  **Environment Step:** It simulates a "Buy" at `decision_date + 1` and calculates the returns over the `holding_period`.
5.  **Reward Generation:** It outputs performance metrics (`holding_p_gain`, `holding_p_sharpe`).

In [16]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
import importlib

modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result"
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import os
import numpy as np

from IPython.display import display
from dataclasses import fields, asdict, is_dataclass
from typing import List, Union, Tuple 


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.engine import AlphaEngine
from core.contracts import MarketObservation, FilterPack
from core.settings import GLOBAL_SETTINGS
from strategy.registry import METRIC_REGISTRY
from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer
from core.paths import OUTPUT_DIR
from core.performance import PerformanceCalculator
from core.features import generate_features
from core.utils import visualize_analyzer_structure, peek
from core.auditor import SystemAuditor

# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output



In [ ]:
# MOVE THIS


def verify_feature_engineering_integrity():
    """
    🛡️ TRIPWIRE: Validates Feature Engineering Logic.
    Enforces:
    1. Day 1 ATR must be NaN (No PrevClose).
    2. Wilder's Smoothing must use Alpha = 1/Period.
    3. Recursion must match manual calculation.
    """
    print("\n--- 🛡️ Starting Feature Engineering Audit ---")

    # 1. Create Synthetic Data (3 Days)
    # Day 1: High-Low = 10. No PrevClose.
    # Day 2: High-Low = 20. Gap up implies TR might be larger.
    # Day 3: High-Low = 10.
    dates = pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
    idx = pd.MultiIndex.from_product([["TEST"], dates], names=["Ticker", "Date"])

    df_mock = pd.DataFrame(
        {
            "Adj Open": [100, 110, 110],
            "Adj High": [110, 130, 120],
            "Adj Low": [100, 110, 110],
            "Adj Close": [105, 120, 115],  # PrevClose: NaN, 105, 120
            "Volume": [1000, 1000, 1000],
        },
        index=idx,
    )

    # 2. Run the Generator
    # We use Period=2 to make manual math easy (Alpha = 1/2 = 0.5)
    feats_df, macro_df = generate_features(
        df_mock, atr_period=2, rsi_period=2, quality_min_periods=1
    )

    atr_series = feats_df["ATR"]

    # 3. MANUAL CALCULATION (The "Truth")
    # Day 1:
    #   TR = Max(H-L, |H-PC|, |L-PC|)
    #   TR = Max(10, NaN, NaN) -> NaN (Because skipna=False)
    #   Expected ATR: NaN

    # Day 2:
    #   PrevClose = 105
    #   H-L=20, |130-105|=25, |110-105|=5
    #   TR = 25
    #   Expected ATR: First valid observation = 25.0

    # Day 3:
    #   PrevClose = 120
    #   H-L=10, |120-120|=0, |110-120|=10
    #   TR = 10
    #   Wilder's Smoothing (Alpha=0.5):
    #   ATR_3 = (TR_3 * alpha) + (ATR_2 * (1-alpha))
    #   ATR_3 = (10 * 0.5) + (25 * 0.5) = 5 + 12.5 = 17.5

    print(f"Audit Values:\n{atr_series.values}")

    # 4. ASSERTIONS
    try:
        # Check Day 1
        if not np.isnan(atr_series.iloc[0]):
            raise AssertionError(
                f"Day 1 Regression: Expected NaN, got {atr_series.iloc[0]}. (Check skipna=False)"
            )

        # Check Day 2 (Initialization)
        if not np.isclose(atr_series.iloc[1], 25.0):
            raise AssertionError(
                f"Initialization Regression: Expected 25.0, got {atr_series.iloc[1]}."
            )

        # Check Day 3 (Recursion)
        if not np.isclose(atr_series.iloc[2], 17.5):
            raise AssertionError(
                f"Wilder's Logic Regression: Expected 17.5, got {atr_series.iloc[2]}. (Check Alpha=1/N)"
            )

        print("✅ FEATURE INTEGRITY PASSED: Wilder's ATR logic is strictly enforced.")

    except AssertionError as e:
        print(f"🔥 LOGIC FAILURE: {str(e)}")
        raise e


#

In [17]:
verify_feature_engineering_integrity()


--- 🛡️ Starting Feature Engineering Audit ---
⚡ Generating Decoupled Features (Benchmark: SPY)...
Audit Values:
[ nan 25.  17.5]
✅ FEATURE INTEGRITY PASSED: Wilder's ATR logic is strictly enforced.


In [28]:
def audit_feature_engineering_integrity(analyzer, df_indices=None, mode="last_run"):
    """
    # Usage to check last run, takes about 4 sec.
    audit_feature_engineering_integrity(analyzer, mode="last_run")
    # Usage to check all df_ohlcv tickers, takes over 4 minutes (i.e. One-time "Nuclear" System Sanity Check)
    audit_feature_engineering_integrity(analyzer, df_indices=df_indices, mode="system")
    """
    import time
    import numpy as np
    import warnings

    # 0. PULL SETTINGS FROM GLOBAL_SETTINGS (or analyzer.engine.settings if stored there)
    # This ensures the auditor uses the EXACT same rules as the engine
    atr_p = GLOBAL_SETTINGS["atr_period"]
    rsi_p = GLOBAL_SETTINGS["rsi_period"]
    win_5 = GLOBAL_SETTINGS["5d_window"]
    win_21 = GLOBAL_SETTINGS["21d_window"]
    win_63 = GLOBAL_SETTINGS["63d_window"]
    q_win = GLOBAL_SETTINGS["quality_window"]
    q_min = GLOBAL_SETTINGS["quality_min_periods"]

    start_time = time.time()
    engine = analyzer.engine
    features_df = engine.features_df
    df_ohlcv = engine.df_ohlcv_raw

    # 1. Scope Selection
    if mode == "last_run" and analyzer.last_run:
        audit_tickers = analyzer.last_run.tickers
        features_to_audit = features_df.loc[pd.IndexSlice[audit_tickers, :], :]
        ohlcv_to_audit = df_ohlcv.loc[pd.IndexSlice[audit_tickers, :], :]
    else:
        audit_tickers = features_df.index.get_level_values(0).unique()
        features_to_audit = features_df
        ohlcv_to_audit = df_ohlcv

    print(f"\n{'='*95}")
    print(
        f"🕵️  NUCLEAR FEATURE AUDIT | Mode: {mode.upper()} | Tickers: {len(audit_tickers)}"
    )
    print(f"{'='*95}")

    # STEP 1: BOUNDARY INTEGRITY
    leaks = features_to_audit.groupby(level=0).head(1)["Ret_1d"].dropna().count()
    leak_status = "✅ PASS" if leaks == 0 else f"❌ FAIL ({leaks} leaks)"
    print(f"STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | {leak_status}")

    # STEP 2: SHADOW CALCULATION
    print(
        f"STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... ", end="", flush=True
    )

    adj_close = ohlcv_to_audit["Adj Close"]
    adj_high = ohlcv_to_audit["Adj High"]
    adj_low = ohlcv_to_audit["Adj Low"]
    volume = ohlcv_to_audit["Volume"]

    shadow_data = {}

    # A. Returns & Basics
    shadow_data["shadow_Ret_1d"] = adj_close.groupby(level=0).pct_change()
    prev_close = adj_close.groupby(level=0).shift(1)
    tr = pd.concat(
        [
            adj_high - adj_low,
            (adj_high - prev_close).abs(),
            (adj_low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1, skipna=False)

    # B. Smoothing (ATR/RSI) - Use transform for speed and index matching
    shadow_data["shadow_ATR"] = tr.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / atr_p, adjust=False).mean()  # Replaced 14
    )

    shadow_data["shadow_ATRP"] = shadow_data["shadow_ATR"] / adj_close
    shadow_data["shadow_TRP"] = tr / adj_close

    # Auditor Step 2B - Shadow RSI with correct Inf/NaN handling
    delta = adj_close.groupby(level=0).diff()
    up, down = delta.clip(lower=0), (-delta).clip(lower=0)

    # Match Wilder's spec correctly:
    roll_up = up.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / rsi_p, adjust=False).mean()  # Replaced 14
    )
    roll_down = down.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / rsi_p, adjust=False).mean()  # Replaced 14
    )

    # FIX: Allow division by zero (i.e. no down day) to create inf (correct RSI=100),
    # inf→100, -inf→0, NaN→50
    # then clean up remaining NaNs (initial periods/no movement)
    # - Initial periods: Before the 14-day lookback is filled, the EWM mean is undefined → NaN.
    # - Flat prices: If price doesn't move (Avg Up = 0 and Avg Down = 0), RS is 0/0 → NaN.
    # - By convention, RSI is set to 50 (neutral) when there is no directional momentum.
    rs = roll_up / roll_down  # Keep zero denominator → inf
    raw_rsi = 100 - (100 / (1 + rs))
    shadow_data["shadow_RSI"] = raw_rsi.replace({np.inf: 100, -np.inf: 0}).fillna(50)

    # C. Momentum & Consistency
    shadow_data[f"shadow_Mom_{win_21}"] = adj_close.groupby(level=0).pct_change(win_21)
    pos_ret = (shadow_data["shadow_Ret_1d"] > 0).astype(float)
    shadow_data["shadow_Consistency"] = pos_ret.groupby(level=0).transform(
        lambda x: x.rolling(win_5).mean()
    )

    # D. Risk (Beta & IR)
    if df_indices is not None:
        try:
            # USE THIS: Pull the single source of truth from the engine
            mkt_ret = engine.macro_df["Mkt_Ret"]
            # Map it to the audit tickers
            mkt_series = mkt_ret.reindex(
                ohlcv_to_audit.index.get_level_values(1)
            ).values
            mkt_series = pd.Series(mkt_series, index=ohlcv_to_audit.index)

            # Shadow Beta
            s_ret = shadow_data["shadow_Ret_1d"]
            shadow_data[f"shadow_Beta_{win_63}"] = (
                s_ret.groupby(level=0)
                .transform(
                    lambda x: x.rolling(win_63).cov(
                        mkt_ret.reindex(x.index.get_level_values(1))
                    )
                    / mkt_ret.reindex(x.index.get_level_values(1)).rolling(win_63).var()
                )
                .fillna(1.0)
            )

            # Shadow IR
            active_ret = s_ret - mkt_series
            shadow_data["shadow_IR_63"] = (
                active_ret.groupby(level=0)
                .transform(lambda x: x.rolling(win_63).mean() / x.rolling(win_63).std())
                .fillna(0.0)
            )

        except Exception as e:
            print(f" (Macro Shadow Error: {e}) ", end="")

    # E. Drawdown & Quality
    roll_max_21 = adj_close.groupby(level=0).transform(
        lambda x: x.rolling(win_21).max()
    )
    shadow_data[f"shadow_DD_{win_21}"] = (adj_close / roll_max_21 - 1).fillna(0.0)
    stale_mask = ((volume == 0) | (adj_high == adj_low)).astype(int)

    shadow_data["shadow_RollingStalePct"] = stale_mask.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).mean()
    )
    dollar_vol = adj_close * volume
    shadow_data["shadow_RollMedDollarVol"] = dollar_vol.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).median()  # Replaced 252, 126
    )

    same_vol = (volume.groupby(level=0).diff() == 0).astype(int)
    shadow_data["shadow_RollingSameVolCount"] = same_vol.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).sum()  # Replaced 252, 126
    )

    # Build Final Shadow DF
    audit_df = pd.DataFrame(shadow_data, index=ohlcv_to_audit.index)
    print(f"DONE ({time.time()-start_time:.2f}s)")

    # STEP 3: RECONCILIATION REPORT
    print(f"\n{'Metric':<20} | {'Max Delta':<12} | {'Correlation':<12} | {'Status'}")
    print("-" * 85)

    cols_to_check = [
        "Ret_1d",
        "ATR",
        "ATRP",
        "TRP",
        "RSI",
        "Mom_21",
        "Consistency",
        "Beta_63",
        "IR_63",
        "DD_21",
        "RollingStalePct",
        "RollMedDollarVol",
        "RollingSameVolCount",
    ]

    for col in cols_to_check:
        sha_col = f"shadow_{col}"
        if sha_col not in audit_df.columns:
            continue

        eng, sha = features_to_audit[col], audit_df[sha_col]
        # Align and drop NaNs for comparison
        mask = eng.notna() & sha.notna()
        if not mask.any():
            continue

        e_v, s_v = eng[mask], sha[mask]

        delta = (e_v - s_v).abs().max()

        # Suppress the NumPy "Subtract" warning during correlation of constant series
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # If standard deviation is 0, correlation is undefined; if eng matches Shadow Calculation, we treat as 1.0
            if e_v.std() == 0:
                corr = 1.0 if delta < 1e-6 else 0.0
            else:
                corr = e_v.corr(s_v)

        status = "✅ PASS" if (delta < 1e-6 or corr > 0.99999) else "❌ FAIL"
        print(f"{col:<20} | {delta:>12.4e} | {corr:>12.6f} | {status}")

    vix_z = engine.macro_df["Macro_Vix_Z"].abs().max()
    print(
        f"{'Macro_Vix_Signals':<20} | {'N/A':<12} | {'N/A':<12} | {'✅ LIVE' if vix_z > 0 else '❌ MISSING VIX, VIX3M'}"
    )
    print(f"{'='*95}")


# def verify_macro_engine(df_ohlcv, df_indices, original_macro_df, settings):
#     """
#     Independently verifies the macro_df calculation logic using GLOBAL_SETTINGS.
#     """
#     print(f"--- Macro Verification (Benchmark: {settings['benchmark_ticker']}) ---")

#     # 1. Setup Skeleton
#     all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
#     v_df = pd.DataFrame(index=all_dates)

#     # Constants from GLOBAL_SETTINGS
#     benchmark = settings["benchmark_ticker"]
#     win_21 = settings["21d_window"]
#     win_63 = settings["63d_window"]
#     z_clip = settings["feature_zscore_clip"]

#     # 2. Market Return & Trend Calculation
#     # Logic: Uses 200-day SMA for the trend anchor
#     if benchmark in df_ohlcv.index.get_level_values("Ticker"):
#         mkt_close = (
#             df_ohlcv.xs(benchmark, level="Ticker")["Adj Close"]
#             .reindex(all_dates)
#             .ffill()
#         )
#         v_df["Mkt_Ret"] = mkt_close.pct_change().fillna(0.0)
#         v_df["Macro_Trend"] = (mkt_close / mkt_close.rolling(200).mean()) - 1.0
#     else:
#         print(f"⚠️ Warning: {benchmark} not found in OHLCV. Defaulting to 0.0.")
#         v_df["Mkt_Ret"] = 0.0
#         v_df["Macro_Trend"] = 0.0

#     # 3. Trend Velocity & Momentum logic
#     v_df["Macro_Trend_Vel"] = v_df["Macro_Trend"].diff(win_21)

#     # Z-Score of Velocity normalized by 63d rolling volatility of the Trend itself
#     v_df["Macro_Trend_Vel_Z"] = (
#         v_df["Macro_Trend_Vel"] / v_df["Macro_Trend"].rolling(win_63).std()
#     ).clip(-z_clip, z_clip)

#     # Momentum: Sign agreement between level and direction
#     v_df["Macro_Trend_Mom"] = (
#         np.sign(v_df["Macro_Trend"])
#         * np.sign(v_df["Macro_Trend_Vel"])
#         * np.abs(v_df["Macro_Trend_Vel"])
#     ).fillna(0)

#     # 4. VIX Engine Logic
#     v_df["Macro_Vix_Z"] = 0.0
#     v_df["Macro_Vix_Ratio"] = 1.0

#     if df_indices is not None:
#         idx_names = df_indices.index.get_level_values(0).unique()
#         if "^VIX" in idx_names:
#             vix = df_indices.xs("^VIX", level=0)["Adj Close"].reindex(all_dates).ffill()
#             # VIX Z-score over 63 days
#             v_df["Macro_Vix_Z"] = (
#                 (vix - vix.rolling(63).mean()) / vix.rolling(63).std()
#             ).clip(-z_clip, z_clip)

#             if "^VIX3M" in idx_names:
#                 vix3m = (
#                     df_indices.xs("^VIX3M", level=0)["Adj Close"]
#                     .reindex(all_dates)
#                     .ffill()
#                 )
#                 v_df["Macro_Vix_Ratio"] = (vix / vix3m).fillna(1.0)

#     # Final cleanup to match original function
#     v_df.fillna(0.0, inplace=True)

#     # 5. Validation Loop
#     print(f"\nComparing verification vs original (Clip Threshold: {z_clip}):")
#     match_all = True
#     for col in original_macro_df.columns:
#         if col not in v_df.columns:
#             print(f"❌ Column '{col}' missing in verification code.")
#             match_all = False
#             continue

#         # Use a tolerance for floating point math
#         diff = np.abs(original_macro_df[col] - v_df[col])
#         max_err = diff.max()

#         if max_err < 1e-9:
#             print(f"✅ {col:<20} | PASS (Max Diff: {max_err:.2e})")
#         else:
#             print(f"⚠️ {col:<20} | FAIL (Max Diff: {max_err:.2e})")
#             match_all = False

#     return v_df if not match_all else None


#

In [31]:
# ==============================================================================
# SECTION H: UTILITIES
# ==============================================================================


def export_debug_to_csv(audit_pack, source_label="Audit"):
    """
    High-Transparency Exporter (Hardened Version).
    Dumps the entire simulation state into a folder for manual Excel verification.
    """
    if not audit_pack or not audit_pack[0]:
        print("❌ Error: Audit Pack is empty. Run a simulation first.")
        return

    data = audit_pack[0]
    # Handle the fact that 'inputs' might be a key or a dataclass attribute
    inputs = data.get("inputs")

    # 1. Folder Setup
    date_str = inputs.start_date.strftime("%Y-%m-%d")
    strat = inputs.metric.replace(" ", "").replace("(", "").replace(")", "")
    folder_name = f"{source_label}_{strat}_{date_str}"

    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    print(f"📂 [AUDIT EXPORT] Folder: ./{folder_name}/")

    def process_item(item, path_prefix=""):
        # A. Handle Nested Dicts
        if isinstance(item, dict):
            for k, v in item.items():
                process_item(v, f"{path_prefix}{k}_" if path_prefix else f"{k}_")

        # B. Handle DataFrames (Matrices - High Precision)
        elif isinstance(item, pd.DataFrame):
            fn = f"Matrix_{path_prefix.strip('_')}.csv"
            item.to_csv(os.path.join(folder_name, fn), float_format="%.8f")
            print(f"   ✅ Matrix: {fn}")

        # C. Handle Series (Vectors)
        elif isinstance(item, pd.Series):
            fn = f"Vector_{path_prefix.strip('_')}.csv"
            item.to_frame().to_csv(os.path.join(folder_name, fn), float_format="%.8f")
            print(f"   ✅ Vector: {fn}")

        # D. Handle Dataclasses (Metadata & Results)
        elif is_dataclass(item):
            class_name = item.__class__.__name__
            fn = f"Summary_{class_name}_{path_prefix.strip('_')}".strip("_") + ".csv"

            # --- THE FIX: Create a Safe Dictionary for Pandas ---
            raw_dict = asdict(item)
            summary_ready_dict = {}

            for k, v in raw_dict.items():
                # If it's a big data object, just note its existence in the summary
                if isinstance(v, (pd.DataFrame, pd.Series)):
                    summary_ready_dict[k] = f"<{v.__class__.__name__} shape={v.shape}>"
                # If it's a list or dict (the crash cause), stringify it for Excel
                elif isinstance(v, (list, dict)):
                    summary_ready_dict[k] = str(v)
                else:
                    summary_ready_dict[k] = v

            # Save the clean key-value summary
            pd.DataFrame.from_dict(
                summary_ready_dict, orient="index", columns=["Value"]
            ).to_csv(os.path.join(folder_name, fn))
            print(f"   📑 Summary: {fn}")

            # E. RECURSION: Now find the actual DataFrames inside the dataclass
            # We iterate the object attributes directly to avoid the 'asdict' list confusion
            for k in item.__dataclass_fields__.keys():
                val = getattr(item, k)
                if isinstance(val, (pd.DataFrame, pd.Series, dict)):
                    process_item(val, f"{path_prefix}{k}_")

    # 3. Execute Extraction
    process_item(data)
    print(f"\n✨ Export Complete. Open ./{folder_name}/ to verify results.")


def export_audit_to_excel(audit_pack, filename="Audit_Verification_Report.xlsx"):
    """
    Final Zero-Base Audit Export.
    Provides everything needed to reconstruct the Strategy results from raw candles.
    Usage: export_audit_to_excel(analyzer1.last_run)
    """
    if audit_pack is None:
        return print("❌ Error: Audit Pack is empty.")

    # Resolve full output path
    output_path = Path(OUTPUT_DIR) / filename

    # Ensure output directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Setup Core References
    debug = audit_pack.debug_data or {}
    inputs = debug.get("inputs_snapshot")
    p_raw = debug.get("portfolio_raw_components", {})
    b_raw = debug.get("benchmark_raw_components", {})

    dec_date = audit_pack.decision_date
    bench_ticker = inputs.benchmark_ticker if inputs else "Benchmark"
    # CHANGED: Export all tickers, not just top 3
    all_tickers = audit_pack.tickers if audit_pack.tickers else []

    print(f"📂 [EXCEL AUDIT] Building full transparency report: {output_path}")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

        # --- SHEET 1: OVERVIEW (Settings & Results) ---
        meta = {**asdict(inputs)} if inputs else {}
        meta.update(audit_pack.perf_metrics or {})
        pd.DataFrame.from_dict(
            {k: str(v) for k, v in meta.items()}, orient="index", columns=["Value"]
        ).to_excel(writer, sheet_name="OVERVIEW")

        # --- SHEET 2: DAILY_AUDIT (The Timeline + Period Labels) ---
        daily = {
            "Port_Value": audit_pack.portfolio_series,
            "Port_ATRP": audit_pack.portfolio_atrp_series,
            "Port_TRP": audit_pack.portfolio_trp_series,
            "Bench_Value": audit_pack.benchmark_series,
            "Bench_ATRP": audit_pack.benchmark_atrp_series,
            "Bench_TRP": audit_pack.benchmark_trp_series,
        }
        if audit_pack.portfolio_series is not None:
            daily["Port_Ret"] = QuantUtils.compute_returns(audit_pack.portfolio_series)
        if audit_pack.benchmark_series is not None:
            daily["Bench_Ret"] = QuantUtils.compute_returns(audit_pack.benchmark_series)

        df_daily = pd.concat({k: v for k, v in daily.items() if v is not None}, axis=1)

        # Add Period Label Column for Excel Range Selection
        df_daily["Period_Label"] = np.where(
            df_daily.index <= dec_date, "LOOKBACK", "HOLDING"
        )
        df_daily.to_excel(writer, sheet_name="DAILY_AUDIT")

        # --- SHEET 3: RAW_OHLCV_SAMPLES (Spot Check for ALL Tickers + Benchmark) ---
        ohlcv_list = []
        # Get Benchmark OHLCV
        if (b_ohlcv := b_raw.get("ohlcv_raw")) is not None:
            b_temp = b_ohlcv.copy()
            b_temp["Ticker"] = bench_ticker
            ohlcv_list.append(b_temp)
        # Get ALL Tickers OHLCV (not just top 3)
        if (p_ohlcv := p_raw.get("ohlcv_raw")) is not None:
            if isinstance(p_ohlcv.index, pd.MultiIndex):
                # CHANGED: Use all_tickers instead of top_3_tickers
                sample_p = p_ohlcv.query("Ticker in @all_tickers")
                ohlcv_list.append(sample_p.reset_index())
            else:
                # Fallback: Filter by 'ticker' column if it exists
                col_name = "ticker" if "ticker" in p_ohlcv.columns else "Ticker"
                if col_name in p_ohlcv.columns:
                    # CHANGED: Use all_tickers instead of top_3_tickers
                    ohlcv_list.append(p_ohlcv[p_ohlcv[col_name].isin(all_tickers)])

        if ohlcv_list:
            pd.concat(ohlcv_list).to_excel(
                writer, sheet_name="RAW_OHLCV_SAMPLES", index=False
            )

        # --- SHEET 4, 5, 6: MERGED MATRICES (Price, ATRP, TRP) ---
        for sheet_name, key in [
            ("RAW_PRICES", "prices"),
            ("RAW_ATRP_DATA", "atrp"),
            ("RAW_TRP_DATA", "trp"),
        ]:
            p_df, b_df = p_raw.get(key), b_raw.get(key)
            if p_df is not None and b_df is not None:
                pd.concat(
                    [p_df, b_df.rename(columns={b_df.columns[0]: bench_ticker})], axis=1
                ).to_excel(writer, sheet_name=sheet_name)

        # --- SHEET 7: RAW_DRIFTED_WEIGHTS ---
        if (p_prices := p_raw.get("prices")) is not None:
            weights_ser = pd.Series(
                audit_pack.initial_weights, index=audit_pack.tickers
            )
            norm_p = p_prices.div(p_prices.bfill().iloc[0])
            weighted = norm_p.mul(weights_ser, axis=1)
            drift_weights = weighted.div(weighted.sum(axis=1), axis=0)
            drift_weights.to_excel(writer, sheet_name="RAW_DRIFTED_WEIGHTS")

        # --- SHEET 8: SURVIVAL_AUDIT (Layer 1 Filter Verification) ---
        if liq_audit := debug.get("audit_liquidity", {}):
            if (snap := liq_audit.get("universe_snapshot")) is not None:
                snap.to_excel(writer, sheet_name="SURVIVAL_AUDIT")

        # --- SHEET 9: FULL_RANKING ---
        if (df_rank := debug.get("full_universe_ranking")) is not None:
            df_rank.to_excel(writer, sheet_name="FULL_RANKING")

    print(f"✨ Audit Report Complete: {output_path}")
    return output_path


def export_last_run_tickers_data_to_csv(
    analyzer, df_ohlcv, features_df, filename="all_tickers_stacked.csv"
):
    """
    Export the last run ticker data from a WalkForwardAnalyzer to a stacked CSV file.

    Parameters:
    -----------
    analyzer : WalkForwardAnalyzer
        The analyzer containing last_run data
    df_ohlcv : pd.DataFrame
        OHLCV data for create_combined_dict
    features_df : pd.DataFrame
        Features data for create_combined_dict
    filename : str
        Output filename (saved to OUTPUT_DIR)

    Returns:
    --------
    Path : Path to saved CSV file
    """

    # 1. Access the result object from the analyzer
    res = analyzer.last_run

    if res is None:
        raise ValueError(
            "❌ No results found in analyzer. Please click 'Run Simulation' first."
        )

    # 2. Extract attributes directly (No [0] needed)
    benchmark = res.debug_data["inputs_snapshot"].benchmark_ticker
    tickers = res.tickers + [benchmark]
    start_date = res.start_date
    end_date = res.holding_end_date  # Note: I use end_date, not decision_date/buy_date

    # 3. Generate the combined dict
    combined = create_combined_dict(
        df_ohlcv=df_ohlcv.copy(),
        features_df=features_df,
        tickers=tickers,
        date_start=start_date,
        date_end=end_date,
        verbose=True,
    )

    # 4. Print ticker data (optional — remove if not needed)
    for ticker in tickers:
        with pd.option_context("display.float_format", "{:.8f}".format):
            print(f"{ticker}:\n{combined[ticker][start_date:end_date]}\n")

    # 5. Save ticker data to CSV
    file_path = filename

    # Save first ticker with header
    first_ticker = tickers[0]
    df_first = combined[first_ticker][start_date:end_date].reset_index()
    df_first["Ticker"] = first_ticker

    df_first.to_csv(file_path, header=True, index=False, lineterminator="\n")
    print(f"✓ Saved {first_ticker} with header")

    # Append remaining tickers without header
    for ticker in tickers[1:]:
        df = combined[ticker][start_date:end_date].reset_index()
        df["Ticker"] = ticker

        df.to_csv(file_path, header=False, index=False, lineterminator="\n", mode="a")
        print(f"✓ Appended {ticker}")

    print(f"\n✓ Saved all tickers to: {file_path}")

    return file_path


def print_nested(d, indent=0, width=4):
    """Pretty-print nested containers.
    Leaves are rendered as two lines:  key\\nvalue ."""
    spacing = " " * indent

    def _kind(node):
        if not isinstance(node, dict):
            return None
        return "sep" if all(isinstance(v, dict) for v in node.values()) else "nest"

    if isinstance(d, dict):
        for k, v in d.items():
            kind = _kind(v)
            tag = "" if kind is None else f"  [{'SEP' if kind == 'sep' else 'NEST'}]"
            print(f"{spacing}{k}{tag}")
            print_nested(v, indent + width, width)

    elif isinstance(d, (list, tuple)):
        for idx, item in enumerate(d):
            print(f"{spacing}[{idx}]")
            print_nested(item, indent + width, width)

    else:  # leaf – primitive value
        print(f"{spacing}{d}")


def get_ticker_OHLCV(
    df_ohlcv: pd.DataFrame,
    tickers: Union[str, List[str]],
    date_start: str,
    date_end: str,
    return_format: str = "dataframe",
    verbose: bool = True,
) -> Union[pd.DataFrame, dict]:
    """
    Get OHLCV data for specified tickers within a date range.

    Parameters
    ----------
    df_ohlcv : pd.DataFrame
        DataFrame with MultiIndex of (ticker, date) and OHLCV columns
    tickers : str or list of str
        Ticker symbol(s) to retrieve
    date_start : str
        Start date in 'YYYY-MM-DD' format
    date_end : str
        End date in 'YYYY-MM-DD' format
    return_format : str, optional
        Format to return data in. Options:
        - 'dataframe': Single DataFrame with MultiIndex (default)
        - 'dict': Dictionary with tickers as keys and DataFrames as values
        - 'separate': List of separate DataFrames for each ticker
    verbose : bool, optional
        Whether to print summary information (default: True)

    Returns
    -------
    Union[pd.DataFrame, dict, list]
        Filtered OHLCV data in specified format

    Raises
    ------
    ValueError
        If input parameters are invalid
    KeyError
        If tickers not found in DataFrame

    Examples
    --------
    >>> # Get data for single ticker
    >>> vlo_data = get_ticker_OHLCV(df_ohlcv, 'VLO', '2025-08-13', '2025-09-04')

    >>> # Get data for multiple tickers
    >>> multi_data = get_ticker_OHLCV(df_ohlcv, ['VLO', 'JPST'], '2025-08-13', '2025-09-04')

    >>> # Get data as dictionary
    >>> data_dict = get_ticker_OHLCV(df_ohlcv, ['VLO', 'JPST'], '2025-08-13',
    ...                              '2025-09-04', return_format='dict')
    """

    # Input validation
    if not isinstance(df_ohlcv, pd.DataFrame):
        raise TypeError("df_ohlcv must be a pandas DataFrame")

    if not isinstance(df_ohlcv.index, pd.MultiIndex):
        raise ValueError("DataFrame must have MultiIndex of (ticker, date)")

    if len(df_ohlcv.index.levels) != 2:
        raise ValueError("MultiIndex must have exactly 2 levels: (ticker, date)")

    # Convert single ticker to list for consistent processing
    if isinstance(tickers, str):
        tickers = [tickers]
    elif not isinstance(tickers, list):
        raise TypeError("tickers must be a string or list of strings")

    # Convert dates to Timestamps
    try:
        start_date = pd.Timestamp(date_start)
        end_date = pd.Timestamp(date_end)
    except ValueError as e:
        raise ValueError(f"Invalid date format. Use 'YYYY-MM-DD': {e}")

    if start_date > end_date:
        raise ValueError("date_start must be before or equal to date_end")

    # Check if tickers exist in the DataFrame
    available_tickers = df_ohlcv.index.get_level_values(0).unique()
    missing_tickers = [t for t in tickers if t not in available_tickers]

    if missing_tickers:
        raise KeyError(f"Ticker(s) not found in DataFrame: {missing_tickers}")

    # Filter the data using MultiIndex slicing
    try:
        filtered_data = df_ohlcv.loc[(tickers, slice(date_start, date_end)), :]
    except Exception as e:
        raise ValueError(f"Error filtering data: {e}")

    # Handle empty results
    if filtered_data.empty:
        if verbose:
            print(
                f"No data found for tickers {tickers} in date range {date_start} to {date_end}"
            )
        return filtered_data

    # Print summary if verbose
    if verbose:
        print(
            f"Data retrieved for {len(tickers)} ticker(s) from {date_start} to {date_end}"
        )
        print(f"Total rows: {len(filtered_data)}")
        print(
            f"Date range in data: {filtered_data.index.get_level_values(1).min()} to "
            f"{filtered_data.index.get_level_values(1).max()}"
        )

        # Print ticker-specific counts
        ticker_counts = filtered_data.index.get_level_values(0).value_counts()
        for ticker in tickers:
            count = ticker_counts.get(ticker, 0)
            if count > 0:
                print(f"  {ticker}: {count} rows")
            else:
                print(f"  {ticker}: No data in range")

    # Return in requested format
    if return_format == "dict":
        result = {}
        for ticker in tickers:
            try:
                result[ticker] = filtered_data.xs(ticker, level=0).loc[
                    date_start:date_end
                ]
            except KeyError:
                result[ticker] = pd.DataFrame()
        return result

    elif return_format == "separate":
        result = []
        for ticker in tickers:
            try:
                result.append(
                    filtered_data.xs(ticker, level=0).loc[date_start:date_end]
                )
            except KeyError:
                result.append(pd.DataFrame())
        return result

    elif return_format == "dataframe":
        return filtered_data

    else:
        raise ValueError(
            f"Invalid return_format: {return_format}. "
            f"Must be 'dataframe', 'dict', or 'separate'"
        )


def get_ticker_features(
    features_df: pd.DataFrame,
    tickers: Union[str, List[str]],
    date_start: str,
    date_end: str,
    return_format: str = "dataframe",
    verbose: bool = True,
) -> Union[pd.DataFrame, dict]:
    """
    Get features data for specified tickers within a date range.

    Parameters
    ----------
    features_df : pd.DataFrame
        DataFrame with MultiIndex of (ticker, date) and feature columns
    tickers : str or list of str
        Ticker symbol(s) to retrieve
    date_start : str
        Start date in 'YYYY-MM-DD' format
    date_end : str
        End date in 'YYYY-MM-DD' format
    return_format : str, optional
        Format to return data in. Options:
        - 'dataframe': Single DataFrame with MultiIndex (default)
        - 'dict': Dictionary with tickers as keys and DataFrames as values
        - 'separate': List of separate DataFrames for each ticker
    verbose : bool, optional
        Whether to print summary information (default: True)

    Returns
    -------
    Union[pd.DataFrame, dict, list]
        Filtered features data in specified format
    """
    # Convert single ticker to list for consistent processing
    if isinstance(tickers, str):
        tickers = [tickers]

    # Filter the data using MultiIndex slicing
    try:
        filtered_data = features_df.loc[(tickers, slice(date_start, date_end)), :]
    except Exception as e:
        if verbose:
            print(f"Error filtering data: {e}")
        return pd.DataFrame() if return_format == "dataframe" else {}

    # Handle empty results
    if filtered_data.empty:
        if verbose:
            print(
                f"No data found for tickers {tickers} in date range {date_start} to {date_end}"
            )
        return filtered_data

    # Print summary if verbose
    if verbose:
        print(
            f"Features data retrieved for {len(tickers)} ticker(s) from {date_start} to {date_end}"
        )
        print(f"Total rows: {len(filtered_data)}")
        print(
            f"Date range in data: {filtered_data.index.get_level_values(1).min()} to "
            f"{filtered_data.index.get_level_values(1).max()}"
        )
        print(f"Available features: {', '.join(filtered_data.columns.tolist())}")

        # Print ticker-specific counts
        ticker_counts = filtered_data.index.get_level_values(0).value_counts()
        for ticker in tickers:
            count = ticker_counts.get(ticker, 0)
            if count > 0:
                print(f"  {ticker}: {count} rows")
            else:
                print(f"  {ticker}: No data in range")

    # Return in requested format
    if return_format == "dict":
        result = {}
        for ticker in tickers:
            try:
                result[ticker] = filtered_data.xs(ticker, level=0).loc[
                    date_start:date_end
                ]
            except KeyError:
                result[ticker] = pd.DataFrame()
        return result

    elif return_format == "separate":
        result = []
        for ticker in tickers:
            try:
                result.append(
                    filtered_data.xs(ticker, level=0).loc[date_start:date_end]
                )
            except KeyError:
                result.append(pd.DataFrame())
        return result

    elif return_format == "dataframe":
        return filtered_data

    else:
        raise ValueError(
            f"Invalid return_format: {return_format}. "
            f"Must be 'dataframe', 'dict', or 'separate'"
        )


def create_combined_dict(
    df_ohlcv: pd.DataFrame,
    features_df: pd.DataFrame,
    tickers: Union[str, List[str]],
    date_start: str,
    date_end: str,
    verbose: bool = True,
) -> dict:
    """
    Create a combined dictionary with both OHLCV and features data for each ticker.

    Parameters:
    -----------
    df_ohlcv : pd.DataFrame
        DataFrame with OHLCV data (MultiIndex: ticker, date)
    features_df : pd.DataFrame
        DataFrame with features data (MultiIndex: ticker, date)
    tickers : str or list of str
        Ticker symbol(s) to retrieve
    date_start : str
        Start date in 'YYYY-MM-DD' format
    date_end : str
        End date in 'YYYY-MM-DD' format
    verbose : bool, optional
        Whether to print progress information (default: True)

    Returns:
    --------
    dict
        Dictionary with tickers as keys and combined DataFrames (OHLCV + features) as values
    """
    # Convert single ticker to list
    if isinstance(tickers, str):
        tickers = [tickers]

    if verbose:
        print(f"Creating combined dictionary for {len(tickers)} ticker(s)")
        print(f"Date range: {date_start} to {date_end}")
        print("=" * 60)

    # Get OHLCV data as dictionary
    ohlcv_dict = get_ticker_OHLCV(
        df_ohlcv, tickers, date_start, date_end, return_format="dict", verbose=verbose
    )

    # Get features data as dictionary
    features_dict = get_ticker_features(
        features_df,
        tickers,
        date_start,
        date_end,
        return_format="dict",
        verbose=verbose,
    )

    # Create combined_dict
    combined_dict = {}

    for ticker in tickers:
        if verbose:
            print(f"\nProcessing {ticker}...")

        # Check if ticker exists in both dictionaries
        if ticker in ohlcv_dict and ticker in features_dict:
            ohlcv_data = ohlcv_dict[ticker]
            features_data = features_dict[ticker]

            # Check if both dataframes have data
            if not ohlcv_data.empty and not features_data.empty:
                # Combine OHLCV and features data
                # Note: Both dataframes have the same index (dates), so we can concatenate
                combined_df = pd.concat([ohlcv_data, features_data], axis=1)

                # Ensure proper index naming
                combined_df.index.name = "Date"

                # Store in combined_dict
                combined_dict[ticker] = combined_df

                if verbose:
                    print(f"  ✓ Successfully combined data")
                    print(f"  OHLCV shape: {ohlcv_data.shape}")
                    print(f"  Features shape: {features_data.shape}")
                    print(f"  Combined shape: {combined_df.shape}")
                    print(
                        f"  Date range: {combined_df.index.min()} to {combined_df.index.max()}"
                    )
            else:
                if verbose:
                    print(f"  ✗ Cannot combine: One or both dataframes are empty")
                    print(f"    OHLCV empty: {ohlcv_data.empty}")
                    print(f"    Features empty: {features_data.empty}")
                combined_dict[ticker] = pd.DataFrame()
        else:
            if verbose:
                print(f"  ✗ Ticker not found in both dictionaries")
                if ticker not in ohlcv_dict:
                    print(f"    Not in OHLCV data")
                if ticker not in features_dict:
                    print(f"    Not in features data")
            combined_dict[ticker] = pd.DataFrame()

    # Print summary
    if verbose:
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"Total tickers processed: {len(tickers)}")

        tickers_with_data = [
            ticker for ticker, df in combined_dict.items() if not df.empty
        ]
        print(f"Tickers with combined data: {len(tickers_with_data)}")

        if tickers_with_data:
            print("\nTicker details:")
            for ticker in tickers_with_data:
                df = combined_dict[ticker]
                print(f"  {ticker}: {df.shape} - {df.index.min()} to {df.index.max()}")
                print(f"    Columns: {len(df.columns)}")

        empty_tickers = [ticker for ticker, df in combined_dict.items() if df.empty]
        if empty_tickers:
            print(f"\nTickers with no data: {', '.join(empty_tickers)}")

    return combined_dict


#

In [19]:
from core.auditor import SystemAuditor

# Automated Audit Suite
checks = [
    SystemAuditor.verify_math_integrity(),
    SystemAuditor.verify_ranking_integrity(),
    SystemAuditor.verify_vol_alignment_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 95)

# Automated Audit Suite
checks = [
    SystemAuditor.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

✅ Mathematical boundaries strictly enforced.
✅ Ranking integrity strictly enforced.
✅ Reward and Risk are strictly synchronized.
--- Macro Verification (Benchmark: SPY) ---

Comparing verification vs original (Clip Threshold: 4.0):
✅ Mkt_Ret              | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend          | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel      | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel_Z    | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Mom      | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Z          | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Ratio      | PASS (Max Diff: 0.00e+00)
✅ Macro Integrity Verified


In [ ]:
from core.auditor import SystemAuditor

# 1. Pre-flight
if not SystemAuditor.verify_math_integrity().ok:
    raise SystemExit("Math Kernels Broken")

# 2. Run Sim
# ... execution ...

# 3. Post-flight Reconciliation
audit = SystemAuditor.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [6]:
data_path = r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet"

df_indices = pd.read_parquet(data_path, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-02-25     20.82     20.92    20.34      20.36       0
       2026-02-26     20.46     21.75    20.41      20.81       0
       2026-02-27     22.19     22.47    21.36      21.56       0
       2026-03-02     23.52     23.52    21.76      22.44       0
       2026-03-03     24.43     26.34    22.78      23.54       0

[144638 rows x 5 columns]


In [7]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

['^AXJO',
 '^BSESN',
 '^DJI',
 '^FCHI',
 '^FTSE',
 '^GDAXI',
 '^GSPC',
 '^HSI',
 '^IXIC',
 '^N225',
 '^NYA',
 '^STOXX50E',
 '^VIX',
 '^VIX3M']

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 144638 entries, ('^AXJO', Timestamp('1992-11-22 00:00:00')) to ('^VIX3M', Timestamp('2026-03-03 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Adj Open   144638 non-null  float64
 1   Adj High   144638 non-null  float64
 2   Adj Low    144638 non-null  float64
 3   Adj Close  144638 non-null  float64
 4   Volume     144638 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 6.6+ MB


In [8]:
print(f"Takes about 1.5 minutes")

data_path = (
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet"
)

df_ohlcv = pd.read_parquet(data_path, engine="pyarrow")

Takes about 1.5 minutes


In [9]:
print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

df_ohlcv.head():
                    Adj Open  Adj High  Adj Low  Adj Close    Volume
Ticker Date                                                        
A      1999-11-18   27.1966   29.8864  23.9091    26.3000  74849965
       1999-11-19   25.6649   25.7023  23.7970    24.1333  18230872
       1999-11-22   24.6936   26.3000  23.9465    26.3000   7871811
       1999-11-23   25.4034   26.0759  23.9091    23.9091   7151082
       1999-11-24   23.9838   25.0672  23.9091    24.5442   5795950

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9494999 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-03 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 399.1+ MB


In [10]:
# === FIXED TEST HARNESS ===
if __name__ == "__main__":
    print("🧪 Testing New Feature Engine...")

    # 1. Create Dummy Index Data
    dates = df_ohlcv.index.get_level_values("Date").unique()
    dummy_indices = []

    for ticker in ["^GSPC", "^VIX", "^VIX3M"]:
        temp = pd.DataFrame(
            {
                "Adj Close": (
                    np.random.normal(100, 5, len(dates))
                    if ticker == "^GSPC"
                    else np.random.normal(20, 2, len(dates))
                ),
                "Volume": 1000,
            },
            index=dates,
        )
        temp["Ticker"] = ticker
        dummy_indices.append(temp.reset_index().set_index(["Ticker", "Date"]))

    df_idx_test = pd.concat(dummy_indices)

    # 2. Run Generation
    try:
        # FIX 1: Unpack the tuple into two variables
        feat_df, mac_df = generate_features(
            df_ohlcv.iloc[:5000], df_indices=df_idx_test
        )

        print("✅ Features Generated Successfully!")

        # FIX 2: Check columns of the features_df (Ticker-specific)
        print("\n--- TICKER FEATURES (Micro) ---")
        print("Columns:", feat_df.columns.tolist())
        # Note: Macro_Vix_Z is no longer in feat_df (that's the point of the optimization!)
        print("Sample Data:\n", feat_df[["Mom_21", "IR_63", "ATRP"]].tail())

        # FIX 3: Check the new Macro DataFrame
        print("\n--- MACRO STATE (Shared) ---")
        print("Columns:", mac_df.columns.tolist())

        # Check specific date alignment
        last_date = feat_df.index.get_level_values("Date")[-1]
        print(f"\n🔍 Macro Check for {last_date.date()}:")
        # We look up the date in the mac_df now
        print(mac_df.loc[last_date])

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback

        traceback.print_exc()

🧪 Testing New Feature Engine...
⚡ Generating Decoupled Features (Benchmark: SPY)...
✅ Features Generated Successfully!

--- TICKER FEATURES (Micro) ---
Columns: ['ATR', 'ATRP', 'TRP', 'RSI', 'Mom_21', 'Consistency', 'IR_63', 'Beta_63', 'DD_21', 'Ret_1d', 'RollingStalePct', 'RollMedDollarVol', 'RollingSameVolCount']
Sample Data:
                    Mom_21   IR_63    ATRP
Ticker Date                              
A      2019-09-27  0.0922  0.0318  0.0195
       2019-09-30  0.0862  0.0208  0.0189
       2019-10-01  0.0547  0.0008  0.0201
       2019-10-02  0.0440 -0.0324  0.0210
       2019-10-03  0.0447 -0.0132  0.0207

--- MACRO STATE (Shared) ---
Columns: ['Mkt_Ret', 'Macro_Trend', 'Macro_Trend_Vel', 'Macro_Trend_Vel_Z', 'Macro_Trend_Mom', 'Macro_Vix_Z', 'Macro_Vix_Ratio']

🔍 Macro Check for 2019-10-03:
Mkt_Ret              0.0000
Macro_Trend          0.0000
Macro_Trend_Vel      0.0000
Macro_Trend_Vel_Z    0.0000
Macro_Trend_Mom      0.0000
Macro_Vix_Z          1.9290
Macro_Vix_Ratio  

In [11]:
# ==============================================================================
# DATA PRE-COMPUTATION (The "Fast-Track" Setup)
# ==============================================================================
print(f"Takes about 2.5 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 2.5 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [12]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9494999 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-03 00:00:00'))
Data columns (total 13 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Consistency          float64
 6   IR_63                float64
 7   Beta_63              float64
 8   DD_21                float64
 9   Ret_1d               float64
 10  RollingStalePct      float64
 11  RollMedDollarVol     float64
 12  RollingSameVolCount  float64
dtypes: float64(13)
memory usage: 978.7+ MB
features_df.info():
None

features_df.index.names:
['Ticker', 'Date']

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16149 entries, 1962-01-02 to 2026-03-03
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------      

In [13]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1581, Days: 16149
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [ ]:
# 1. Remove the old reference from memory
try:
    # del peek
    # del visualize_analyzer_structure
    # del visualize_audit_structure
    # del verify_analyzer_short
    # del verify_analyzer_long
    del core.SystemAuditor
    print("✅ Old local functions removed.")
except NameError:
    print("ℹ️ Functions were already cleared or not found.")

# 2. Re-import the fresh versions from the modular file
# from core.utils import visualize_analyzer_structure, peek
# from.core.auditor import SystemAuditor

# print("🚀 Modular utils imported successfully.")

In [14]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 937 stocks passed filters on 2025-12-10


In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [ ]:
###############################
###############################

In [21]:
my_analyzer = analyzer1

my_res = visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(259,))
[  2]   📈 benchmark_series (shape=(259,))
[  3]   🧮 normalized_plot_data (shape=(259, 80))
[  4]   📂 tickers (len=80)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]     📄 index_10 (str)
[ 16]     📄 index_11 (str)
[ 17]     📄 index_12 (str)
[ 18]     📄 index_13 (str)
[ 19]     📄 index_14 (str)
[ 20]     📄 index_15 (str)
[ 21]     📄 index_16 (str)
[ 22]     📄 index_17 (str)
[ 23]     📄 index_18 (str)
[ 24]     📄 index_19 (str)
[ 25]     📄 index_20 (str)
[ 26]     📄 index_21 (str)
[ 27]     📄 index_22 (str)
[ 28]     📄 index_23 (str)
[ 29]     📄 index_24 (str)
[ 30]     📄 index_25 (str)
[ 31]     📄 index_26 (str)
[ 32]     📄 index_27 (str)
[ 33]     📄 index_28 (str)
[ 3

In [22]:
peek(58, my_res)

 📍 INDEX: [58]
 🏷️  NAME:  index_53
 📂 PATH:  audit_pack -> tickers -> index_53



'NEM'

'NEM'

In [23]:
# 3. Post-flight Reconciliation
audit = SystemAuditor.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here


***********************************************************************************************
🕵️  STARTING SHORT-FORM AUDIT: Sharpe (TRP) @ 2026-02-23
⚠️  ASSUMPTION: Verification logic is independent, but trusts Engine source DataFrames
   (engine.features_df, engine.df_close, and debug['portfolio_raw_components'])
***********************************************************************************************
🕵️  AUDIT: Sharpe (TRP) @ 2026-02-23
LAYER 1: SURVIVAL  | Universe: 1580 -> Survivors: 942 | ✅ PASS
LAYER 2: SELECTION | Strategy: Sharpe (TRP) | Selection Match: ✅ PASS
LAYER 3: PERFORMANCE (Holding Period: 5 days)
Metric               | Engine       | Manual       | Status
-----------------------------------------------------------------------------------------------
Gain                 |    -0.034453 |    -0.034453 | ✅ PASS
Sharpe               |    -4.982838 |    -4.982838 | ✅ PASS
Sharpe (ATRP)        |    -0.231625 |    -0.231625 | ✅ PASS
Sharpe (TRP)         |    -0.18

In [24]:
audit = SystemAuditor.verify_analyzer_long(my_analyzer, n_tickers=5)


🛡️  STARTING NUCLEAR AUDIT | 2026-02-23 | Sharpe (TRP)
📝 1. PERFORMANCE RECONCILIATION


Period                     Full Holding Lookback
Entity    Metric                                
Benchmark Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS
Group     Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS


📝 2. SURVIVAL AUDIT
   Survival Integrity: ✅ PASS (Engine: 942 vs Auditor: 942)

📝 3. UNIVERSAL SELECTION AUDIT | Strategy: Sharpe (TRP)
   Scope: Evaluated 942 candidates (Full Universe).
   Result: 942 PASSED | 0 FAILED
   All scores match registry math. Sharpe (TRP) results of the first 5 tickers


,Ticker,Engine,Manual,Delta,Status
Rank,,,,,
1,A,-0.00531600,-0.00531600,0.00000000,✅ PASS
2,AA,0.05371587,0.05371587,0.00000000,✅ PASS
3,AAL,-0.00467275,-0.00467275,0.00000000,✅ PASS
4,AAPL,0.02184251,0.02184251,0.00000000,✅ PASS
5,ABBV,0.03266328,0.03266328,0.00000000,✅ PASS


In [25]:
import importlib
import core.auditor
from core.auditor import SystemAuditor

# 1. Force Python to reload the file from disk
importlib.reload(core.auditor)

# 2. Re-import the class specifically
from core.auditor import SystemAuditor

print("✅ SystemAuditor reloaded with 'n_tickers' support.")

# 3. Try the call again
audit = SystemAuditor.verify_analyzer_long(my_analyzer, n_tickers=20)

✅ SystemAuditor reloaded with 'n_tickers' support.

🛡️  STARTING NUCLEAR AUDIT | 2026-02-23 | Sharpe (TRP)
📝 1. PERFORMANCE RECONCILIATION


Period                     Full Holding Lookback
Entity    Metric                                
Benchmark Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS
Group     Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS


📝 2. SURVIVAL AUDIT
   Survival Integrity: ✅ PASS (Engine: 942 vs Auditor: 942)

📝 3. UNIVERSAL SELECTION AUDIT | Strategy: Sharpe (TRP)
   Scope: Evaluated 942 candidates (Full Universe).
   Result: 942 PASSED | 0 FAILED
   All scores match registry math. Sharpe (TRP) results of the first 20 tickers


,Ticker,Engine,Manual,Delta,Status
Rank,,,,,
1,A,-0.00531600,-0.00531600,0.00000000,✅ PASS
2,AA,0.05371587,0.05371587,0.00000000,✅ PASS
3,AAL,-0.00467275,-0.00467275,0.00000000,✅ PASS
4,AAPL,0.02184251,0.02184251,0.00000000,✅ PASS
5,ABBV,0.03266328,0.03266328,0.00000000,✅ PASS
6,ABNB,-0.02262472,-0.02262472,0.00000000,✅ PASS
7,ABT,-0.01842508,-0.01842508,0.00000000,✅ PASS
8,ACGL,0.02446844,0.02446844,0.00000000,✅ PASS
9,ACI,-0.00495916,-0.00495916,0.00000000,✅ PASS


In [29]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
audit_feature_engineering_integrity(my_analyzer, mode="last_run")


🕵️  NUCLEAR FEATURE AUDIT | Mode: LAST_RUN | Tickers: 80
STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | ✅ PASS
STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... DONE (4.98s)

Metric               | Max Delta    | Correlation  | Status
-------------------------------------------------------------------------------------
Ret_1d               |   0.0000e+00 |     1.000000 | ✅ PASS
ATR                  |   0.0000e+00 |     1.000000 | ✅ PASS
ATRP                 |   0.0000e+00 |     1.000000 | ✅ PASS
TRP                  |   0.0000e+00 |     1.000000 | ✅ PASS
RSI                  |   0.0000e+00 |     1.000000 | ✅ PASS
Mom_21               |   0.0000e+00 |     1.000000 | ✅ PASS
Consistency          |   0.0000e+00 |     1.000000 | ✅ PASS
DD_21                |   0.0000e+00 |     1.000000 | ✅ PASS
RollingStalePct      |   0.0000e+00 |     1.000000 | ✅ PASS
RollMedDollarVol     |   0.0000e+00 |     1.000000 | ✅ PASS
RollingSameVolCount  |   0.0000e+00 |     1.000000 | ✅ PASS


In [32]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

### Export Ticker's OHLCV and its Features

In [33]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

Creating combined dictionary for 81 ticker(s)
Date range: 2025-02-20 00:00:00 to 2026-03-03 00:00:00
Data retrieved for 81 ticker(s) from 2025-02-20 00:00:00 to 2026-03-03 00:00:00
Total rows: 20979
Date range in data: 2025-02-20 00:00:00 to 2026-03-03 00:00:00
  SGOV: 259 rows
  SHV: 259 rows
  BIL: 259 rows
  MINT: 259 rows
  BOXX: 259 rows
  USFR: 259 rows
  PULS: 259 rows
  JPST: 259 rows
  VGSH: 259 rows
  SHY: 259 rows
  EWY: 259 rows
  VCSH: 259 rows
  SNDK: 259 rows
  BSV: 259 rows
  JAAA: 259 rows
  IGSB: 259 rows
  GLDM: 259 rows
  SGOL: 259 rows
  IAU: 259 rows
  LITE: 259 rows
  EFV: 259 rows
  GLD: 259 rows
  WDC: 259 rows
  SLV: 259 rows
  EMXC: 259 rows
  AU: 259 rows
  PHYS: 259 rows
  GLW: 259 rows
  PSLV: 259 rows
  TD: 259 rows
  STX: 259 rows
  BTI: 259 rows
  HII: 259 rows
  SIL: 259 rows
  GDXJ: 259 rows
  GDX: 259 rows
  FIX: 259 rows
  VEA: 259 rows
  MU: 259 rows
  SPDW: 259 rows
  KGC: 259 rows
  CIEN: 259 rows
  CAT: 259 rows
  HSBC: 259 rows
  SATS: 259 rows

### Audit features_df

In [ ]:
# # Takes 4 minutes to run, checks all tickers from my_analyzer
# audit_feature_engineering_integrity(my_analyzer, df_indices=df_indices, mode="system")